# Q14: scan_v2 point-in-time universe robustness

Compare the current 47-symbol historical backfill (optimistic upper bound), a universe frozen from information visible on 2023-03-01, and a monthly rolling point-in-time universe. Signal, execution, exit, cost, and portfolio rules remain identical.

In [ ]:
import os
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'xquant').exists():
            return candidate
    raise RuntimeError('repository root not found')

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'scan-v2-right-side-5-pit-universe'
TRADING_ROOT = REPO_ROOT.parent / 'trading'

## 1. Reproduce the locked comparison

The broad point-in-time decision stream is generated with the production replay in the trading repository. The xquant script then applies pre-registered eligibility, liquidity ranking, and market caps.

In [ ]:
import subprocess

subprocess.run([
    'uv', 'run', '--project', str(TRADING_ROOT / 'backend'), 'python',
    'scripts/replay_scan_v2_bull_flip_events.py',
    '--replay-script', str(TRADING_ROOT / 'scripts' / 'replay_unified_decisions.py'),
    '--data-dir', str(TRADING_ROOT / 'backend' / 'data'),
    '--output', str(OUTPUT_DIR / 'broad_decision_stream.parquet'),
], check=True)

subprocess.run([
    'uv', 'run', 'python', 'scripts/research_scan_v2_pit_universe.py',
    '--current-decisions', 'outputs/unified-scan-policy-v2-breakout-r5/decision_stream.parquet',
    '--broad-decisions', str(OUTPUT_DIR / 'broad_decision_stream.parquet'),
    '--price-data-dir', '../trading/backend/data',
    '--output-dir', str(OUTPUT_DIR),
], check=True)

## 2. Q3/Q5 scorecard and descriptive CAGR decomposition

In [ ]:
import pandas as pd

metrics = pd.read_csv(OUTPUT_DIR / 'portfolio_metrics.csv')
attribution = pd.read_csv(OUTPUT_DIR / 'cagr_attribution.csv')
display(metrics[['signal', 'universe', 'cagr', 'sharpe', 'calmar', 'maxDrawdown', 'maxRecoveryDays', 'annualizedTurnover', 'averageGrossExposure']])
display(attribution)

## 3. Annual, monthly, and market behavior

In [ ]:
annual = pd.read_csv(OUTPUT_DIR / 'annual_metrics.csv')
monthly = pd.read_csv(OUTPUT_DIR / 'monthly_metrics.csv')
by_market = pd.read_csv(OUTPUT_DIR / 'portfolio_metrics_by_market.csv')
display(annual)
display(monthly.groupby(['signal', 'universe']).agg(months=('return', 'size'), monthly_win_rate=('return', lambda x: (x > 0).mean()), best_month=('return', 'max'), worst_month=('return', 'min')).reset_index())
display(by_market[['signal', 'universe', 'market', 'cagr', 'sharpe', 'maxDrawdown', 'averageGrossExposure', 'closedTrades']])

## 4. Universe history, removals, and predefined stress checks

In [ ]:
membership = pd.read_csv(OUTPUT_DIR / 'universe_membership_history.csv')
stress = pd.read_csv(OUTPUT_DIR / 'universe_sensitivity.csv')
display(membership.groupby(['universe', 'effectiveDate', 'market'])['selected'].sum().reset_index())
display(membership[membership['membershipEvent'].str.startswith('exited')])
display(stress[['scenario', 'signal', 'averageSelectedSymbols', 'cagr', 'sharpe', 'maxDrawdown']])

## 5. Conclusion discipline

Use the written review note for the final interpretation. The CAGR decomposition is descriptive rather than causal because the local cache is not a complete historical exchange security master. No signal or universe parameter is selected from these full-sample outcomes.